# Visualize Dynamic Object Filter Output 
## 1. Imports and Setup

In [1]:
import sys
import os
import numpy as np
import time
import yaml # For loading the single config file
import json # Keep just in case, but yaml is primary
import k3d
import matplotlib.colors as mcolors
import ipywidgets
from pyquaternion import Quaternion

# --- Add NuScenes Devkit ---
try:
    from nuscenes.nuscenes import NuScenes
    from nuscenes.utils.data_classes import LidarPointCloud
    NUSCENES_AVAILABLE = True
    print("NuScenes SDK imported successfully.")
except ImportError as e:
    print(f"Warning: nuScenes-devkit import failed: {e}")
    NUSCENES_AVAILABLE = False

# --- Add mpy_detector module path ---
# Assuming notebook is run from the repository root or similar
repo_root = os.path.abspath('.') # Adjust if notebook is elsewhere
build_dir = os.path.join(repo_root, 'build')
if build_dir not in sys.path:
     print(f"Adding build directory to sys.path: {build_dir}")
     sys.path.insert(0, build_dir)

try:
    import mpy_detector as mdet
    print("Successfully imported mpy_detector.")
except ImportError as e:
    print(f"ERROR: Could not import mpy_detector from {build_dir}: {e}")
    # Optional: Exit or raise error if module is essential
    mdet = None # Set to None to handle gracefully later


NuScenes SDK imported successfully.
Adding build directory to sys.path: /home/drugge/Unsupervised-Moving-Point-Detection/build
Successfully imported mpy_detector.


## 2. Configuration Loading

In [2]:


# --- Define Paths ---
# Assumes config is in 'test_data/configs/' relative to repo root
CONFIG_FILENAME = "test_full_config.yaml"
config_path = os.path.join(repo_root, 'test_data', 'configs', CONFIG_FILENAME)

# --- Load YAML Config ---
py_config = {}
if os.path.exists(config_path):
    print(f"Loading configuration from: {config_path}")
    try:
        with open(config_path, 'r') as f:
            py_config = yaml.safe_load(f)
        if py_config is None:
            print("Warning: Config file is empty or invalid.")
            py_config = {}
        else:
            print("Config loaded successfully.")
            # print(json.dumps(py_config, indent=2)) # Optional: Print loaded config
    except Exception as e:
        print(f"Error loading config file {config_path}: {e}")
else:
    print(f"ERROR: Configuration file not found at {config_path}")

# --- Extract Dataset Info ---
nuscenes_version = None
nuscenes_dataroot = None
if 'dataset' in py_config and isinstance(py_config['dataset'], dict):
    dataset_config = py_config['dataset']
    nuscenes_version = dataset_config.get('version')
    dataroot_raw = dataset_config.get('dataroot')
    if dataroot_raw:
        try:
            nuscenes_dataroot = os.path.expanduser(dataroot_raw)
            print(f"NuScenes Version: {nuscenes_version}")
            print(f"NuScenes Dataroot (Expanded): {nuscenes_dataroot}")
            if not os.path.exists(nuscenes_dataroot):
                 print(f"Warning: Expanded dataroot does not exist: {nuscenes_dataroot}")
        except Exception as e:
            print(f"Error expanding dataroot path '{dataroot_raw}': {e}")
    else:
         print("Warning: 'dataroot' not found in dataset config.")
else:
    print("Warning: 'dataset' section not found or invalid in config.")



Loading configuration from: /home/drugge/Unsupervised-Moving-Point-Detection/test_data/configs/test_full_config.yaml
Config loaded successfully.
NuScenes Version: v1.0-mini
NuScenes Dataroot (Expanded): /datasets/nuscenes


## 3. Helper Functions

In [3]:

def get_sensor_global_pose(nusc, lidar_data_token):
    """Calculates the global pose of the LIDAR sensor for a given sample_data token."""
    lidar_data = nusc.get('sample_data', lidar_data_token)
    sensor_record = nusc.get('calibrated_sensor', lidar_data['calibrated_sensor_token'])
    pose_record = nusc.get('ego_pose', lidar_data['ego_pose_token'])

    # Combine sensor calibration and ego pose
    ego_quat = Quaternion(pose_record['rotation'])
    sensor_calib_quat = Quaternion(sensor_record['rotation'])
    final_quat = ego_quat * sensor_calib_quat # Global Sensor Rot = Global Ego Rot * Ego Sensor Rot

    ego_translation = np.array(pose_record['translation'])
    sensor_calib_translation = np.array(sensor_record['translation'])
    # Rotate sensor translation to global frame and add ego translation
    final_translation = ego_translation + ego_quat.rotate(sensor_calib_translation)

    rotation_matrix = final_quat.rotation_matrix.astype(np.float64)
    position_vector = final_translation.astype(np.float64)

    return rotation_matrix, position_vector

def plot_axes_numpy(plot, T_plotorigin_target=np.eye(4), length=1.0, origin_label=""):
    """Adds k3d axes representation using NumPy."""
    unit_vectors = np.array([[length, 0, 0, 1], [0, length, 0, 1], [0, 0, length, 1]])
    start_points = np.tile(T_plotorigin_target[:3, 3], (3, 1)) # Repeat origin 3 times
    end_points_h = (T_plotorigin_target @ unit_vectors.T).T # Apply transform
    end_points = end_points_h[:, :3] # Get non-homogeneous coords

    # Define colors for X(Red), Y(Green), Z(Blue)
    colors = [0xff0000, 0xff0000, 0x00ff00, 0x00ff00, 0x0000ff, 0x0000ff]

    plot += k3d.vectors(origins=start_points, vectors=(end_points - start_points), colors=colors)
    if origin_label:
         plot += k3d.text(origin_label, position=start_points[0] + np.array([-0.1, -0.1, -0.1]), color=0xffffff, size=0.5)


def create_label_colormap(num_labels=7):
    """Creates a dictionary mapping integer labels to hex color codes."""
    # Use a distinct colormap like 'tab10' or generate programmatically
    cmap = mcolors.ListedColormap(mcolors.TABLEAU_COLORS) # tab10
    # cmap = mcolors.get_cmap('viridis', num_labels) # Or 'jet', 'hsv', etc.
    colormap = {}
    label_names = { # Map enum value to string name
        0: "STATIC", 1: "CASE1", 2: "CASE2", 3: "CASE3",
        4: "SELF", 5: "UNCERTAIN", 6: "INVALID"
    }
    for i in range(num_labels):
        rgb = cmap(i)[:3] # Get RGB tuple (0-1 range)
        hex_color = int(mcolors.to_hex(rgb)[1:], 16) # Convert to 0xRRGGBB integer
        colormap[i] = {'hex': hex_color, 'name': label_names.get(i, f"LBL_{i}")}
    return colormap

def plot_point_cloud_with_labels(plot, points_xyz, labels=None, colormap=None, default_color=0xaaaaaa,
                                 point_size=0.1, offset=[0,0,0], text_indices=None, text_prefix=""):
    """Adds points to a k3d plot, colored by labels or a default color. Optionally adds text labels."""
    if points_xyz is None or len(points_xyz) == 0:
        return

    points_to_plot = points_xyz + np.array(offset) # Apply offset for side-by-side plotting

    if labels is not None and colormap is not None and len(labels) == len(points_xyz):
        # Generate colors based on labels
        colors = np.array([colormap.get(lbl, {'hex': default_color})['hex'] for lbl in labels], dtype=np.uint32)
        plot += k3d.points(positions=points_to_plot.astype(np.float32), colors=colors, point_size=point_size)
    else:
        # Use default color
        plot += k3d.points(positions=points_to_plot.astype(np.float32), color=default_color, point_size=point_size)

    # Add text labels for specific indices
    if text_indices is not None:
        for i in text_indices:
            if 0 <= i < len(points_to_plot):
                point_pos = points_to_plot[i]
                label_val = labels[i] if labels is not None and i < len(labels) else -1
                label_info = colormap.get(label_val, {'name': 'N/A'}) if colormap else {'name': 'N/A'}
                label_name = label_info['name']
                text = f"{text_prefix}{i} ({label_name})"
                plot += k3d.text(text, position=point_pos + np.array([0.1, 0.1, 0.1]), color=0xffffff, size=0.3)


## 4. Initialization

In [4]:
# --- Initialize NuScenes ---
nusc = None
if NUSCENES_AVAILABLE and nuscenes_dataroot and nuscenes_version:
    try:
        nusc = NuScenes(version=nuscenes_version, dataroot=nuscenes_dataroot, verbose=False)
        print(f"NuScenes initialized. Found {len(nusc.scene)} scenes.")
    except Exception as e:
        print(f"Error initializing NuScenes: {e}")
else:
    print("Skipping NuScenes initialization.")

# --- Initialize C++ Filter ---
dyn_filter = None
if mdet and os.path.exists(config_path):
    try:
        print(f"Initializing DynObjFilter using C++ config: {config_path}")
        dyn_filter = mdet.DynObjFilter(config_path=config_path)
        print("DynObjFilter initialized successfully.")
    except Exception as e:
        print(f"Error initializing DynObjFilter: {e}")
else:
    if not mdet:
        print("Skipping DynObjFilter initialization (module not imported).")
    else:
        print(f"Skipping DynObjFilter initialization (C++ config not found: {config_path}).")



NuScenes initialized. Found 10 scenes.
Initializing DynObjFilter using C++ config: /home/drugge/Unsupervised-Moving-Point-Detection/test_data/configs/test_full_config.yaml
DynObjFilter initialized successfully.
Pybind: Constructing DynObjFilter with config: /home/drugge/Unsupervised-Moving-Point-Detection/test_data/configs/test_full_config.yaml
DynObjFilter constructed. History length: 10
  RingBuffer capacity: 10
DynObjFilter initialization complete.


## 5. Select Scene/Sample and Process

In [5]:



# --- Parameters ---
SCENE_INDEX = 0 # Index of the scene to load
SAMPLE_INDEX = 0 # Index of the sample (sweep) within the scene
POINTS_TO_LABEL = list(range(7)) # Indices of points to add text labels for
PLOT_OFFSET = [5, 0, 0] # Offset for the classified point cloud plot

# --- Data Loading and Processing ---
points_np = None
labels_np = None
points_sensor_frame = None
scene_name = "N/A"
sample_token = "N/A"

if nusc and dyn_filter and SCENE_INDEX < len(nusc.scene):
    scene_info = nusc.scene[SCENE_INDEX]
    scene_name = scene_info['name']
    print(f"\nProcessing Scene: {scene_name} (Index {SCENE_INDEX})")

    if SAMPLE_INDEX < scene_info['nbr_samples']:
        sample_token = nusc.field2token('sample', 'scene_token', scene_info['token'])[SAMPLE_INDEX]
        sample_record = nusc.get('sample', sample_token)
        lidar_token = sample_record['data']['LIDAR_TOP']
        lidar_data = nusc.get('sample_data', lidar_token)
        timestamp_usec = lidar_data['timestamp']
        scan_timestamp_sec = timestamp_usec / 1e6

        # Load point cloud
        lidar_filepath = nusc.get_sample_data_path(lidar_token)
        print(f"  Loading Sample Index {SAMPLE_INDEX}, Lidar Token: {lidar_token}")
        print(f"  Filepath: {lidar_filepath}")
        pc = LidarPointCloud.from_file(lidar_filepath)
        points_sensor_frame = pc.points.T # Keep in sensor frame (X,Y,Z,Intensity)

        # Get Global Pose for C++ filter
        rotation_matrix, position_vector = get_sensor_global_pose(nusc, lidar_token)

        # Prepare data for C++ filter (ensure correct types)
        points_np_input = points_sensor_frame[:, :4].astype(np.float32) # Use Nx4

        # Call C++ Filter
        print(f"  Calling filter for {points_np_input.shape[0]} points...")
        try:
            start_time = time.time()
            labels_np = dyn_filter.process_scan_placeholder(
                points_np=points_np_input,
                rotation=rotation_matrix,
                position=position_vector,
                timestamp=scan_timestamp_sec
            )
            end_time = time.time()
            print(f"  Filter returned {labels_np.shape[0]} labels in {end_time - start_time:.4f} seconds.")

            # Basic validation
            if labels_np.shape[0] != points_np_input.shape[0]:
                print(f"  ERROR: Number of labels ({labels_np.shape[0]}) does not match number of points ({points_np_input.shape[0]})!")
                labels_np = None # Invalidate labels

        except Exception as e:
            print(f"  ERROR calling filter: {e}")
            labels_np = None

    else:
        print(f"ERROR: Sample index {SAMPLE_INDEX} out of range for scene {scene_name} (max {scene_info['nbr_samples']-1}).")
else:
    if not nusc: print("Cannot process: NuScenes not initialized.")
    if not dyn_filter: print("Cannot process: DynObjFilter not initialized.")
    if nusc and SCENE_INDEX >= len(nusc.scene): print(f"Cannot process: Scene index {SCENE_INDEX} out of range.")



Processing Scene: scene-0061 (Index 0)
  Loading Sample Index 0, Lidar Token: 9d9bf11fb0e144c8b446d54a8a00184f
  Filepath: /datasets/nuscenes/samples/LIDAR_TOP/n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402927647951.pcd.bin
  Calling filter for 34688 points...
  ERROR calling filter: 'mpy_detector.DynObjFilter' object has no attribute 'process_scan_placeholder'


## 6. Visualization

In [ ]:

plot = None # Define plot variable outside the if block

if points_sensor_frame is not None:
    print("\nCreating K3D plot...")
    label_colormap = create_label_colormap()
    # print("Label Colormap:", label_colormap) # Optional: print the generated map

    plot = k3d.plot(camera_auto_fit=False, grid_visible=True, menu_visibility=True)

    # Plot Sensor Origin Axes
    plot_axes_numpy(plot, np.eye(4), length=1.0, origin_label="Sensor")

    # Plot Original Point Cloud (Grey) - No offset
    print("  Adding original point cloud...")
    plot_point_cloud_with_labels(
        plot,
        points_xyz=points_sensor_frame[:, :3], # Use only XYZ
        labels=None, # No labels for original
        default_color=0x888888, # Grey color
        point_size=0.05,
        offset=[0, 0, 0],
        text_indices=POINTS_TO_LABEL,
        text_prefix="Orig "
    )

    # Plot Classified Point Cloud (Colored) - With offset
    if labels_np is not None:
        print("  Adding classified point cloud...")
        plot_point_cloud_with_labels(
            plot,
            points_xyz=points_sensor_frame[:, :3], # Use only XYZ
            labels=labels_np,
            colormap=label_colormap,
            default_color=0xff00ff, # Magenta for errors/missing labels
            point_size=0.05,
            offset=PLOT_OFFSET, # Apply offset here
            text_indices=POINTS_TO_LABEL,
            text_prefix="Class "
        )
        # Add axes for the offset cloud origin for clarity
        offset_transform = np.eye(4)
        offset_transform[:3, 3] = PLOT_OFFSET
        plot_axes_numpy(plot, offset_transform, length=0.5, origin_label="Classified")

    else:
        print("  Skipping classified point cloud plot (labels not available).")

    # Set initial camera view (optional, adjust as needed)
    plot.camera = [10, -10, 5, 0, 0, 0, 0, 0, 1] # Example view
    print("Displaying plot...")
    # display(plot) # Display command is implicit in last line of cell

else:
    print("No point cloud data loaded to visualize.")

# Display the plot if it was created
plot



Creating K3D plot...
  Adding original point cloud...
  Adding classified point cloud...
Displaying plot...


/home/drugge/miniconda3/envs/mdet_env/lib/python3.7/site-packages/traittypes/traittypes.py:101: UserWarning: Given trait value dtype "float64" does not match required type "float32". A coerced copy has been created.
  np.dtype(self.dtype).name))


Plot(antialias=3, axes=['x', 'y', 'z'], axes_helper=1.0, axes_helper_colors=[16711680, 65280, 255], background…

In [ ]:

# ## 7. Interactive Widgets (Optional)

# This section allows selecting scene/sample interactively.
# You would need to wrap the processing and plotting logic (sections 5 & 6)
# into a function and call it using ipywidgets.interact.

# Example structure:

# def run_and_visualize(scene_idx, sample_idx):
#     global plot # Allow modifying the global plot object
#     print(f"--- Running for Scene {scene_idx}, Sample {sample_idx} ---")
#     # --- Repeat logic from Section 5 ---
#     # Load data for scene_idx, sample_idx
#     # Call filter
#     # --- Repeat logic from Section 6 ---
#     # Create plot
#     # Add points (original and classified)
#     # Set camera
#     plot = k3d_plot_object # Assign the created plot
#     display(plot) # Explicit display needed inside function

# if nusc:
#     scene_widget = ipywidgets.Dropdown(options=range(len(nusc.scene)), description='Scene:')
#     # Initialize sample widget based on first scene
#     initial_samples = range(nusc.scene[0]['nbr_samples']) if len(nusc.scene) > 0 else []
#     sample_widget = ipywidgets.Dropdown(options=initial_samples, description='Sample:')

#     def update_sample_options(*args):
#         scene_idx = scene_widget.value
#         if scene_idx < len(nusc.scene):
#             sample_widget.options = range(nusc.scene[scene_idx]['nbr_samples'])
#         else:
#             sample_widget.options = []
#     scene_widget.observe(update_sample_options, 'value')

#     ipywidgets.interact(run_and_visualize, scene_idx=scene_widget, sample_idx=sample_widget)
# else:
#     print("Skipping interactive widgets (NuScenes not initialized).")